In [2]:
# ============================================================
# INSTALL (jalankan sekali jika di Colab baru)
# ============================================================
!pip install -q transformers datasets sentencepiece torch evaluate rouge_score

# ============================================================
# IMPORT LIBRARY
# ============================================================
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    TrainingArguments,
    Trainer
)
import evaluate

# ============================================================
# DATASET DUMMY
# ============================================================
data = {
    "article": [
        "Timnas Indonesia berhasil mengalahkan Thailand dengan skor 2-1 pada final AFF 2025 yang berlangsung di Jakarta.",

        "Pemerintah mengumumkan kenaikan anggaran pendidikan sebesar 15 persen untuk meningkatkan kualitas sekolah.",

        "Harga emas dunia mengalami kenaikan akibat meningkatnya ketidakpastian ekonomi global.",

        "Universitas Muhammadiyah Malang mengadakan seminar kecerdasan buatan yang dihadiri ratusan mahasiswa.",

        "BMKG memperkirakan hujan lebat akan terjadi di beberapa wilayah Jawa Timur selama tiga hari ke depan."
    ],

    "summary": [
        "Indonesia menang 2-1 atas Thailand di final AFF.",

        "Anggaran pendidikan naik 15 persen.",

        "Harga emas naik karena ketidakpastian ekonomi.",

        "UMM menggelar seminar kecerdasan buatan.",

        "BMKG memprediksi hujan lebat di Jawa Timur."
    ]
}

df = pd.DataFrame(data)

# ============================================================
# CONVERT TO DATASET
# ============================================================
dataset = Dataset.from_pandas(df)

# ============================================================
# LOAD MODEL
# ============================================================
model_name = "t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# ============================================================
# PREPROCESSING
# ============================================================
def preprocess_function(examples):

    inputs = [
        "summarize: " + article
        for article in examples["article"]
    ]

    model_inputs = tokenizer(
        inputs,
        max_length=256,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        examples["summary"],
        max_length=64,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True
)

# ============================================================
# TRAIN TEST SPLIT
# ============================================================
train_dataset = tokenized_dataset.select([0, 1, 2, 3])
test_dataset = tokenized_dataset.select([4])

# ============================================================
# TRAINING CONFIG
# ============================================================
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

# ============================================================
# TRAIN MODEL
# ============================================================
print("Training model...")
trainer.train()

# ============================================================
# TEST ARTICLE
# ============================================================
test_text = """
BMKG menyatakan bahwa hujan dengan intensitas tinggi
diperkirakan terjadi di Jawa Timur selama tiga hari ke depan.
Masyarakat diminta waspada terhadap banjir dan tanah longsor.
"""

input_ids = tokenizer(
    "summarize: " + test_text,
    return_tensors="pt"
).input_ids

output = model.generate(
    input_ids,
    max_length=50,
    num_beams=4
)

generated_summary = tokenizer.decode(
    output[0],
    skip_special_tokens=True
)

print("\n==============================")
print("HASIL RINGKASAN")
print("==============================")
print(generated_summary)

# ============================================================
# ROUGE EVALUATION
# ============================================================
reference_summary = "BMKG memperkirakan hujan lebat terjadi di Jawa Timur."

rouge = evaluate.load("rouge")

scores = rouge.compute(
    predictions=[generated_summary],
    references=[reference_summary]
)

print("\n==============================")
print("ROUGE SCORE")
print("==============================")
for key, value in scores.items():
    print(f"{key}: {value:.4f}")


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
Map: 100%|██████████| 5/5 [00:00<00:00, 652.77 examples/s]


ImportError: Using the `Trainer` with `PyTorch` requires `accelerate>=1.1.0`: Please run `pip install transformers[torch]` or `pip install 'accelerate>=1.1.0'`